# ECG Arrhythmia Benchmark (v2) — Master Colab Notebook

Reproduces every number and figure in:
*Accuracy versus Interpretability in Inter-Patient ECG Arrhythmia Classification: A Controlled Benchmark of Six Model Families with Temporally-Aligned SHAP Attribution on MIT-BIH and PTB-XL* (IEEE Access 2026, submitted).

**How to run:** *Runtime → Change runtime type → T4 GPU (optional) → Run all.*

Pipeline: install → clone → download MIT-BIH → preprocess → train 6 models → evaluate → significance → SHAP (unified) → Grad-CAM → ablation (A1–A4) → PTB-XL (optional) → zip outputs.

## 1. Install dependencies

In [ ]:
# Setup: point Python at the uploaded files.
# Works whether you uploaded src/ as a folder OR dropped the .py files flat into /content/.
import os, sys, shutil
from pathlib import Path

ROOT = Path('/content') if Path('/content').exists() else Path.cwd()
SRC  = ROOT / 'src'

# Modules that must live inside src/ for `from src.X import Y` to work.
SRC_FILES = ['__init__.py', 'config.py', 'data.py', 'models.py',
             'train.py', 'evaluate.py', 'explain.py',
             'ablation.py', 'ptbxl.py']

# Case 1: src/ already present -> nothing to do.
# Case 2: files were dropped flat into ROOT -> move them into src/.
if not SRC.exists():
    flat = [f for f in SRC_FILES if (ROOT / f).exists()]
    if flat:
        SRC.mkdir(parents=True, exist_ok=True)
        for f in flat:
            shutil.move(str(ROOT / f), str(SRC / f))
        print(f'Moved {len(flat)} flat files into {SRC}/')
    else:
        raise FileNotFoundError(
            f'Could not find src/ or flat module files in {ROOT}. '
            'Upload src/ (or its .py files) via the Colab file browser.'
        )

# Ensure every required module is now inside src/
missing = [f for f in SRC_FILES if not (SRC / f).exists()]
if missing:
    raise FileNotFoundError(f'Missing from src/: {missing}')

# Guarantee it's importable as a package
init = SRC / '__init__.py'
if not init.exists():
    init.write_text('')

os.environ['ECG_V2_ROOT'] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

for sub in ('outputs/tables', 'outputs/figures', 'outputs/models',
            'data/raw', 'data/clean'):
    (ROOT / sub).mkdir(parents=True, exist_ok=True)

# Clear any stale cached import
for m in [k for k in list(sys.modules) if k == 'src' or k.startswith('src.')]:
    del sys.modules[m]

print('ROOT        =', ROOT)
print('src/ files  =', sorted(p.name for p in SRC.iterdir()))
print('data/ found =', (ROOT / 'data').exists())

**Upload checklist** (do this once, then `Runtime → Run all`):

1. Upload this notebook to Colab.
2. In the Colab file browser (folder icon, left sidebar), drag-drop the entire `src/` folder into `/content/`.
3. (Optional) If you have a prebuilt `data/` folder, drop it into `/content/` too — otherwise the next cells will auto-download MIT-BIH from PhysioNet via `wfdb`.
4. Run all cells from the top. All outputs land in `/content/outputs/` and are zipped in the final cell.

In [ ]:
# (deprecated clone cell — setup now happens in cell 2)
pass

## 3. Imports and environment check

In [ ]:
import numpy as np
import pandas as pd
import torch

from src.config   import (log, set_global_seed, RANDOM_SEED,
                          AAMI_CLASSES, NUM_CLASSES, WINDOW_SIZE,
                          TABLES, FIGURES, MODELS_DIR, OUTPUTS)
from src.data     import download_mitbih, preprocess_mitbih
from src.models   import (ResNet1D, CNN1D, BiLSTM,
                          build_random_forest, build_svm, build_xgboost,
                          DL_MODELS, ML_MODELS, count_parameters)
from src.train    import train_dl, train_ml, predict_dl, predict_ml, get_device
from src.evaluate import (compute_metrics, per_class_f1,
                          plot_confusion_matrix, plot_roc,
                          plot_model_comparison, plot_per_class_f1,
                          wilcoxon_correctness, mcnemar_test, save_table)
from src.explain  import (shap_temporal_deep, shap_beeswarm_tree,
                          grad_cam_resnet1d)
from src.ablation import run_ablation

set_global_seed(RANDOM_SEED)
print('device =', get_device(), '| torch =', torch.__version__)

## 4. Download MIT-BIH and preprocess (DS1/DS2)

In [ ]:
download_mitbih()
X_tr, y_tr, X_te, y_te = preprocess_mitbih(force=False)
print(f'DS1 train: X={X_tr.shape}, y dist={np.bincount(y_tr, minlength=NUM_CLASSES)}')
print(f'DS2 test : X={X_te.shape}, y dist={np.bincount(y_te, minlength=NUM_CLASSES)}')

## 5. Train all six model families

Identical preprocessing, class weighting, and seed across families. Deep models use Adam + ReduceLROnPlateau + early stopping.

In [ ]:
# --- Deep models ---
trained = {}
for name, builder in DL_MODELS.items():
    log.info(f'=== training {name} ===')
    m = builder()
    train_dl(m, X_tr, y_tr, X_te, y_te, epochs=50, tag=name)
    trained[name] = m
    log.info(f'{name} params = {count_parameters(m):,}')

# --- Classical ML models ---
for name, builder in ML_MODELS.items():
    log.info(f'=== fitting {name} ===')
    trained[name] = train_ml(builder(), X_tr, y_tr, tag=name)

list(trained.keys())

## 6. Predict and compute metrics for every model

In [ ]:
preds = {}     # name -> (y_pred, y_prob)
rows  = []

for name, model in trained.items():
    if name in DL_MODELS:
        y_pred, y_prob = predict_dl(model, X_te)
    else:
        y_pred, y_prob = predict_ml(model, X_te)
    preds[name] = (y_pred, y_prob)

    m  = compute_metrics(y_te, y_pred, y_prob)
    pc = per_class_f1(y_te, y_pred)
    rows.append({'model': name, **m, **pc,
                 'params': count_parameters(model) if name in DL_MODELS else np.nan})

overall_df = pd.DataFrame(rows).sort_values('f1_M', ascending=False).reset_index(drop=True)
save_table(overall_df, 'overall')
overall_df

## 7. Per-model confusion matrices and ROC curves

In [ ]:
for name, (y_pred, y_prob) in preds.items():
    plot_confusion_matrix(y_te, y_pred, tag=name, normalize=True)
    plot_roc(y_te, y_prob, tag=name)

plot_model_comparison(overall_df)
plot_per_class_f1(overall_df)

## 8. Statistical significance: Wilcoxon + McNemar + effect sizes

At n≈49,693 any non-trivial disagreement gives p ≈ 0, so we also record McNemar discordant counts (b01, b10) and the signed disagreement — this is the practically meaningful measure.

In [ ]:
ref = 'ResNet1D'
ref_pred, _ = preds[ref]

sig_rows = []
for name, (y_pred, _) in preds.items():
    if name == ref:
        continue
    p_wil = wilcoxon_correctness(ref_pred, y_pred, y_te)
    p_mc, b01, b10 = mcnemar_test(ref_pred, y_pred, y_te)
    sig_rows.append({
        'comparison':       f'{ref} vs {name}',
        'p_wilcoxon':       p_wil,
        'p_mcnemar':        p_mc,
        f'{ref}_correct_other_wrong': b01,
        f'{ref}_wrong_other_correct': b10,
        'net_advantage':    b01 - b10,
    })

sig_df = pd.DataFrame(sig_rows)
save_table(sig_df, 'significance')
sig_df

## 9. Unified SHAP attribution

Deep models → `GradientExplainer` → per-class mean |φ| over the 260-sample beat.
Tree / classical models → `TreeExplainer` → beeswarm over the same 260-sample input space.
Both live on the same axis so they are directly comparable.

In [ ]:
# --- Deep: temporal attribution per class ---
profiles = {}
for name in ['ResNet1D', 'CNN1D', 'BiLSTM']:
    log.info(f'SHAP temporal -> {name}')
    profiles[name] = shap_temporal_deep(
        trained[name], X_tr, X_te, y_te, tag=name,
        n_background=128, per_class_samples=200)

# --- Tree / classical: beeswarm over sample-index feature space ---
for name in ['RandomForest', 'XGBoost']:
    log.info(f'SHAP tree -> {name}')
    shap_beeswarm_tree(trained[name], X_tr, X_te, tag=name,
                      n_background=200, n_explain=2000)

## 10. Grad-CAM for the 1D-ResNet

Complementary spatial attention over the final residual block; should broadly agree with SHAP temporal attribution.

In [ ]:
grad_cam_resnet1d(trained['ResNet1D'], X_te, y_te, tag='resnet1d')

## 11. Ablation study (A1 – A4)

Four variants of the 1D-ResNet trained from scratch with identical seed and hyperparameters:
* A1 – NoResidual · A2 – Shallow (2 blocks) · A3 – NoBatchNorm · A4 – NoDropout.

In [ ]:
ablation_df = run_ablation(X_tr, y_tr, X_te, y_te, epochs=40)
ablation_df

## 12. (Optional) PTB-XL 12-lead cross-dataset evaluation

Downloads ~1.7 GB. Skip this cell for a MIT-BIH-only run. Enable for the full paper reproduction.

In [ ]:
RUN_PTBXL = False  # set True to run PTB-XL (adds ~30 min on Colab)

if RUN_PTBXL:
    from src.ptbxl import run_ptbxl
    ptbxl_df = run_ptbxl(epochs=20)
    display(ptbxl_df)
else:
    print('PTB-XL skipped (set RUN_PTBXL = True to enable).')

## 13. Package all outputs into a single zip

In [ ]:
import shutil
archive = shutil.make_archive(str(OUTPUTS.parent / 'outputs'), 'zip', OUTPUTS)
print('Wrote', archive)
print('Tables :', sorted(p.name for p in TABLES.glob('*.csv')))
print('Figures:', sorted(p.name for p in FIGURES.glob('*.png')))
print('Models :', sorted(p.name for p in MODELS_DIR.glob('*')))

In [ ]:
# Optional: trigger browser download in Colab
try:
    from google.colab import files
    files.download(archive)
except Exception as e:
    print('files.download unavailable (not on Colab?):', e)